In [ ]:
print("Task 3.3 - Classification Modeling")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Load dataset
df = pd.read_csv("../data/cleaned_data.csv")

# 2. Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Columns in dataset:")
print(df.columns.tolist())

# 3. Convert date columns if they exist
if 'order_date' in df.columns:
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    df['order_month'] = df['order_date'].dt.month
    df['order_day'] = df['order_date'].dt.day
    df['order_hour'] = df['order_date'].dt.hour

if 'shipping_date' in df.columns:
    df['shipping_date'] = pd.to_datetime(df['shipping_date'], errors='coerce')

if 'order_date' in df.columns and 'shipping_date' in df.columns:
    df['shipping_time_days'] = (df['shipping_date'] - df['order_date']).dt.days

# 4. Drop unnecessary columns if they exist
drop_cols = [
    'order_id',
    'customer_id',
    'product_id',
    'order_date',
    'shipping_date',
    'shipping_address',
    'billing_address'
]

df = df.drop(columns=drop_cols, errors='ignore')

# 5. Check target column
target = 'delivery_status'

if target not in df.columns:
    raise ValueError(f"'{target}' column not found. Available columns: {df.columns.tolist()}")

# 6. Remove rows where target is missing
df = df.dropna(subset=[target])

# 7. Define features and target
X = df.drop(columns=[target], errors='ignore')
y = df[target]

# 8. Detect categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

print("\nCategorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)
print("\nTarget classes:")
print(y.value_counts())

# 9. Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# 10. Full model pipeline
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='lbfgs', class_weight='balanced'))
])

# 11. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 12. Train model
clf.fit(X_train, y_train)

# 13. Predictions
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

# 14. Evaluation
print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# 15. Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
plt.title("Confusion Matrix - Delivery Status")
plt.show()

# 16. Show class probabilities
print("\nModel classes:")
print(clf.classes_)

print("\nFirst 5 predicted probability rows:")
print(pd.DataFrame(y_proba[:5], columns=clf.classes_))

Markdown - Customer segmentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# -----------------------------
# 1. Load and prepare dataset
# -----------------------------
df = pd.read_csv("../data/cleaned_data.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Columns:")
print(df.columns.tolist())

# Convert dates if available
if 'order_date' in df.columns:
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    df['order_month'] = df['order_date'].dt.month
    df['order_day'] = df['order_date'].dt.day
    df['order_hour'] = df['order_date'].dt.hour

if 'shipping_date' in df.columns:
    df['shipping_date'] = pd.to_datetime(df['shipping_date'], errors='coerce')

if 'order_date' in df.columns and 'shipping_date' in df.columns:
    df['shipping_time_days'] = (df['shipping_date'] - df['order_date']).dt.days

# Drop columns that are not useful for modeling
drop_cols = [
    'order_id', 'customer_id', 'product_id',
    'order_date', 'shipping_date',
    'shipping_address', 'billing_address'
]
df = df.drop(columns=drop_cols, errors='ignore')

# -----------------------------
# 2. Set target
# -----------------------------
target = 'customer_segment'

if target not in df.columns:
    raise ValueError(f"'{target}' not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=[target])

X = df.drop(columns=[target], errors='ignore')
y = df[target]

# -----------------------------
# 3. Identify feature types
# -----------------------------
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

print("\nCategorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)
print("\nTarget distribution:")
print(y.value_counts())

# -----------------------------
# 4. Preprocessing
# -----------------------------
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# -----------------------------
# 5. Build classification model
# -----------------------------
customer_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='lbfgs', class_weight='balanced'))
])

# -----------------------------
# 6. Split data
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -----------------------------
# 7. Train model
# -----------------------------
customer_model.fit(X_train, y_train)

# -----------------------------
# 8. Predict
# -----------------------------
y_pred = customer_model.predict(X_test)
y_proba = customer_model.predict_proba(X_test)

# -----------------------------
# 9. Evaluate
# -----------------------------
print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=customer_model.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
plt.title("Confusion Matrix - Customer Segment")
plt.show()

# -----------------------------
# 10. Show probabilities
# -----------------------------
print("\nClasses:")
print(customer_model.classes_)

print("\nFirst 5 predicted probability rows:")
print(pd.DataFrame(y_proba[:5], columns=customer_model.classes_))

In [ ]:
import numpy as np

classes = list(customer_model.classes_)
vip_index = classes.index('VIP') if 'VIP' in classes else None

if vip_index is not None:
    custom_pred = []

    for probs in y_proba:
        if probs[vip_index] >= 0.30:
            custom_pred.append('VIP')
        else:
            custom_pred.append(classes[np.argmax(probs)])

    print("Classification Report with VIP threshold = 0.30")
    print(classification_report(y_test, custom_pred))

    cm_custom = confusion_matrix(y_test, custom_pred, labels=classes)
    print("Confusion Matrix with VIP threshold:")
    print(cm_custom)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm_custom, display_labels=classes)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
    plt.title("Customer Segment - Threshold Adjusted for VIP")
    plt.show()
else:
    print("VIP class not found in target labels.")